Structure of the script allows creation of English and Russian versions of the table. But only Russian direction is fully completed, since English table for health regions has already existed and I don't see reason to do redundant work.

Data sources: [data for health regions](https://www150.statcan.gc.ca/t1/tbl1/en/tv.action?pid=1310038901) (till 2015-2017), [archived version](https://www150.statcan.gc.ca/t1/tbl1/en/tv.action?pid=1310006301) (till 2014-2016)

In [2]:
import pandas as pd
import math
import re
from collections import namedtuple
import json

import sys
sys.path.append("..")
import mal_moduls_private.mal_total as mal

In [3]:
# load data
df = pd.read_csv('data/Canada_Health_Regions_till_2017.zip', compression='zip',
                 usecols = ['REF_DATE', 'GEO', 'Age group', 'Sex', 'Characteristics', 'VALUE'])

df = df.loc[(df['Age group'] == 'At birth') & (df['Characteristics'] == 'Life expectancy') & 
             df['REF_DATE'].isin(['2010/2012', '2015/2017']) & ~df['GEO'].str.startswith('Peer group ') & ~df['GEO'].str.endswith(' Health Integration Network, Ontario')]  

print(f"Number of records before removing duplicates: {len(df)}")

df.drop_duplicates(keep='first', inplace=True)
print(f"Number of records after removing duplicates : {len(df)}")

df.head(10)

Number of records before removing duplicates: 768
Number of records after removing duplicates : 747


,REF_DATE,GEO,Age group,Sex,Characteristics,VALUE
10872,2010/2012,Canada,At birth,Both sexes,Life expectancy,81.7
10875,2010/2012,Canada,At birth,Males,Life expectancy,79.4
10878,2010/2012,Canada,At birth,Females,Life expectancy,83.8
10890,2010/2012,Newfoundland and Labrador,At birth,Both sexes,Life expectancy,79.5
10893,2010/2012,Newfoundland and Labrador,At birth,Males,Life expectancy,77.2
10896,2010/2012,Newfoundland and Labrador,At birth,Females,Life expectancy,81.9
10908,2010/2012,"Eastern Regional Health Authority, Newfoundlan...",At birth,Both sexes,Life expectancy,79.8
10911,2010/2012,"Eastern Regional Health Authority, Newfoundlan...",At birth,Males,Life expectancy,77.6
10914,2010/2012,"Eastern Regional Health Authority, Newfoundlan...",At birth,Females,Life expectancy,82.0
10926,2010/2012,"Central Regional Health Authority, Newfoundlan...",At birth,Both sexes,Life expectancy,79.8


In [4]:
df = df.drop(columns=['Age group', 'Characteristics']) \
       .rename(columns={'REF_DATE': 'year',
                        'GEO': 'region',
                        'Sex': 'sex',
                        'VALUE': 'value'})
df

,year,region,sex,value
10872,2010/2012,Canada,Both sexes,81.7
10875,2010/2012,Canada,Males,79.4
10878,2010/2012,Canada,Females,83.8
10890,2010/2012,Newfoundland and Labrador,Both sexes,79.5
10893,2010/2012,Newfoundland and Labrador,Males,77.2
...,...,...,...,...
26931,2015/2017,Northwest Territories,Males,75.2
26934,2015/2017,Northwest Territories,Females,79.3
26964,2015/2017,Nunavut,Both sexes,72.1
26967,2015/2017,Nunavut,Males,70.8


In [5]:
df['health_region'] = df['region'].map(lambda st: ', '.join(st.split(', ')[:-1]) if ', ' in st else f'{st} on average')
df['province'] = df['region'].map(lambda st: st.split(', ')[-1].strip() if ', ' in st else st)

del df['region']

df['health_region'] = df['health_region'].replace({
    'Zone 1 - Western' : 'Western zone',
    'Zone 2 - Northern' : 'Northern zone',
    'Zone 1 - Western' : 'Western zone',
    'Zone 2 - Northern' : 'Northern zone',
    'Zone 3 - Eastern' : 'Eastern zone',
    'Zone 4 - Central' : 'Central zone',
    'Zone 1 (Moncton area)' : 'Moncton area',
    'Zone 2 (Saint John area)' : 'Saint John area',
    'Zone 3 (Fredericton area)' : 'Fredericton area',
    'Zone 4 (Edmundston area)' : 'Edmundston area',
    'Zone 5 (Campbellton area)' : 'Campbellton area',
    'Zone 6 (Bathurst area)' : 'Bathurst area',
    'Zone 7 (Miramichi area)' : 'Miramichi area',
    'Ontario by Local Health Integration Network on average' : 'Ontario on average',
    'Ontario by Health Unit on average' : 'Ontario on average'
})

df['province'] = df['province'].replace({
    'Ontario by Local Health Integration Network' : 'Ontario',
    'Ontario by Health Unit' : 'Ontario'
})

df.drop_duplicates(keep='first', inplace=True)

print(f"Number of records after removing duplicates : {len(df)}")

df.head(10)

Number of records after removing duplicates : 741


,year,sex,value,health_region,province
10872,2010/2012,Both sexes,81.7,Canada on average,Canada
10875,2010/2012,Males,79.4,Canada on average,Canada
10878,2010/2012,Females,83.8,Canada on average,Canada
10890,2010/2012,Both sexes,79.5,Newfoundland and Labrador on average,Newfoundland and Labrador
10893,2010/2012,Males,77.2,Newfoundland and Labrador on average,Newfoundland and Labrador
10896,2010/2012,Females,81.9,Newfoundland and Labrador on average,Newfoundland and Labrador
10908,2010/2012,Both sexes,79.8,Eastern Regional Health Authority,Newfoundland and Labrador
10911,2010/2012,Males,77.6,Eastern Regional Health Authority,Newfoundland and Labrador
10914,2010/2012,Females,82.0,Eastern Regional Health Authority,Newfoundland and Labrador
10926,2010/2012,Both sexes,79.8,Central Regional Health Authority,Newfoundland and Labrador


<br>
<br>

In [7]:
# pd.options.display.max_rows = 300
# pd.options.display.max_colwidth = 80

<br>
<br>

In [9]:
df_total = df.loc[df['sex'] == 'Both sexes'] \
             .pivot(index=['health_region', 'province'], columns='year', values='value') \
             .reset_index('province', drop=False)

df_total.index.name = df_total.columns.name = ''

print(df_total.shape)
df_total.fillna('')

(124, 3)


,province,2010/2012,2015/2017
,,,
Alberta on average,Alberta,81.4,81.5
Bathurst area,New Brunswick,82.1,81.4
Brant County Health Unit,Ontario,79.5,80.2
British Columbia on average,British Columbia,82.4,82.4
Calgary Zone,Alberta,83.1,83.1
...,...,...,...
Western zone,Nova Scotia,81.1,80.9
Windsor-Essex County Health Unit,Ontario,81.4,81.7
Winnipeg Regional Health Authority,Manitoba,80.6,80.8


In [10]:
df_male = df.loc[df['sex'] == 'Males'] \
             .pivot(index=['health_region', 'province'], columns='year', values='value') \
             .reset_index('province', drop=False)

df_male.index.name = df_male.columns.name = ''

print(df_male.shape)
df_male.fillna('')

(124, 3)


,province,2010/2012,2015/2017
,,,
Alberta on average,Alberta,79.1,79.3
Bathurst area,New Brunswick,79.2,79.2
Brant County Health Unit,Ontario,77.1,78.1
British Columbia on average,British Columbia,80.3,80.1
Calgary Zone,Alberta,81.0,81.2
...,...,...,...
Western zone,Nova Scotia,79.1,78.6
Windsor-Essex County Health Unit,Ontario,79.1,79.7
Winnipeg Regional Health Authority,Manitoba,78.4,78.6


In [11]:
df_female = df.loc[df['sex'] == 'Females'] \
             .pivot(index=['health_region', 'province'], columns='year', values='value') \
             .reset_index('province', drop=False)

df_female.index.name = df_female.columns.name = ''

print(df_female.shape)
df_female.fillna('')

(124, 3)


,province,2010/2012,2015/2017
,,,
Alberta on average,Alberta,83.6,83.8
Bathurst area,New Brunswick,84.8,83.6
Brant County Health Unit,Ontario,81.7,82.2
British Columbia on average,British Columbia,84.4,84.7
Calgary Zone,Alberta,85.1,85.0
...,...,...,...
Western zone,Nova Scotia,83.1,83.1
Windsor-Essex County Health Unit,Ontario,83.6,83.7
Winnipeg Regional Health Authority,Manitoba,82.6,82.8


<br>
<br>

In [13]:
df_regions = pd.concat([
        df_total['province'],
        df_total['2010/2012'], df_male['2010/2012'], df_female['2010/2012'], (df_female['2010/2012']-df_male['2010/2012']).round(2),
        (df_total['2015/2017']-df_total['2010/2012']).round(2),
        df_total['2015/2017'], df_male['2015/2017'], df_female['2015/2017'], (df_female['2015/2017']-df_male['2015/2017']).round(2)],
    axis='columns',
    keys=['province', '2010-12_t', '2010-12_m', '2010-12_f', '2010-12_fΔm', 'Δ_periods', '2015-17_t', '2015-17_m', '2015-17_f', '2015-17_fΔm']
)

# del df, df_total, df_male, df_female

# df_regions.dropna(how='all', inplace=True)

df_regions.sort_values(['2015-17_t', '2015-17_m', '2015-17_f'], ascending=False, inplace=True)
df_regions = pd.concat([df_regions.loc[['Canada on average']], df_regions.drop(index='Canada on average')])

df_regions.fillna('')

,province,2010-12_t,2010-12_m,2010-12_f,2010-12_fΔm,Δ_periods,2015-17_t,2015-17_m,2015-17_f,2015-17_fΔm
,,,,,,,,,,
Canada on average,Canada,81.7,79.4,83.8,4.4,0.4,82.1,80.0,84.1,4.1
Richmond Health Service Delivery Area,British Columbia,85.5,84.0,86.7,2.7,1.3,86.8,85.2,88.2,3.0
York Regional Health Unit,Ontario,85.1,83.5,86.6,3.1,0.6,85.7,84.0,87.1,3.1
Peel Regional Health Unit,Ontario,84.4,82.4,86.1,3.7,0.4,84.8,83.0,86.4,3.4
Halton Regional Health Unit,Ontario,83.6,81.8,85.1,3.3,0.9,84.5,82.6,86.2,3.6
...,...,...,...,...,...,...,...,...,...,...
Mamawetan/Keewatin/Athabasca,Saskatchewan,73.4,70.8,76.6,5.8,-0.1,73.3,71.0,75.8,4.8
Northern Regional Health Authority,Manitoba,72.6,70.4,75.0,4.6,0.1,72.7,70.3,75.2,4.9
Nunavut on average,Nunavut,71.7,69.3,74.4,5.1,0.4,72.1,70.8,73.4,2.6


<br>
<br>

In [15]:
# Group by countries and sort sections by the first record in them
sorted_provinces = df_regions.groupby('province').first().sort_values(by=['2015-17_t', '2015-17_m', '2015-17_f', '2010-12_t', '2010-12_m'], ascending=False).index
sorted_provinces

df_regions = pd.concat([df_regions[df_regions['province'] == province] for province in sorted_provinces])
df_regions

,province,2010-12_t,2010-12_m,2010-12_f,2010-12_fΔm,Δ_periods,2015-17_t,2015-17_m,2015-17_f,2015-17_fΔm
,,,,,,,,,,
Richmond Health Service Delivery Area,British Columbia,85.5,84.0,86.7,2.7,1.3,86.8,85.2,88.2,3.0
Vancouver Health Authority,British Columbia,84.4,82.2,86.4,4.2,0.1,84.5,82.1,86.8,4.7
Vancouver Health Service Delivery Area,British Columbia,84.0,81.6,86.3,4.7,0.1,84.1,81.6,86.6,5.0
North Shore/Coast Garibaldi Health Service Delivery Area,British Columbia,84.2,82.2,86.1,3.9,-0.3,83.9,81.6,86.2,4.6
Fraser North Health Service Delivery Area,British Columbia,83.3,81.3,85.1,3.8,0.3,83.6,81.6,85.6,4.0
...,...,...,...,...,...,...,...,...,...,...
Newfoundland and Labrador on average,Newfoundland and Labrador,79.5,77.2,81.9,4.7,0.0,79.5,77.5,81.4,3.9
Labrador-Grenfell Regional Health Authority,Newfoundland and Labrador,79.1,75.6,83.1,7.5,-1.2,77.9,75.7,80.3,4.6
Northwest Territories on average,Northwest Territories,77.9,75.4,80.2,4.8,-0.8,77.1,75.2,79.3,4.1


<br>
<br>

In [17]:
mal.min_and_max_values(df_regions.loc[:, '2010-12_t':], max_lng=15, row_center='Canada on average')

Number of records: 124


,2010-12_t,2010-12_m,2010-12_f,2010-12_fΔm,Δ_periods,2015-17_t,2015-17_m,2015-17_f,2015-17_fΔm
max,85.5 -Richmond Healt…,84.0 -Richmond Healt…,86.7 -Richmond Healt…,7.5 -Labrador-Grenf…,1.8 -Région du Nord…,86.8 -Richmond Healt…,85.2 -Richmond Healt…,88.2 -Richmond Healt…,7.4 -Région du Nuna…
max_2,85.1 -York Regional …,83.5 -York Regional …,86.6 -York Regional …,6.9 -Timiskaming He…,1.5 -Région des Ter…,85.7 -York Regional …,84.0 -York Regional …,87.1 -York Regional …,6.4 -Miramichi area
max_3,84.4 -Vancouver Heal…,82.4 -Peel Regional …,86.4 -Vancouver Heal…,6.4 -Kelsey Trail R…,1.3 -Richmond Healt…,84.8 -Peel Regional …,83.0 -Peel Regional …,86.8 -Vancouver Heal…,6.2 -Prince Albert …
Canada on average,– 81.7 –,– 79.4 –,– 83.8 –,– 4.4 –,– 0.4 –,– 82.1 –,– 80.0 –,– 84.1 –,– 4.1 –
min_3,72.6 -Northern Regio…,70.4 -Northern Regio…,75.0 -Northern Regio…,3.1 -York Regional …,-0.8 -Central Zone,72.7 -Northern Regio…,70.8 -Nunavut on ave…,75.2 -Northern Regio…,2.6 -Nunavut on ave…
min_2,71.7 -Nunavut on ave…,69.3 -Nunavut on ave…,74.4 -Nunavut on ave…,3.0 -Région du Nord…,-1.2 -Labrador-Grenf…,72.1 -Nunavut on ave…,70.3 -Northern Regio…,73.4 -Nunavut on ave…,2.6 -Région de la C…
min,68.4 -Région du Nuna…,66.1 -Région du Nuna…,70.3 -Région du Nuna…,2.7 -Richmond Healt…,-1.5 -Miramichi area,68.7 -Région du Nuna…,65.6 -Région du Nuna…,73.0 -Région du Nuna…,2.1 -Région du Nord…


<br>
<br>

In [19]:
# for region in sorted(df_regions.index.to_list()):
#     print(f"    '{region}': {{'en': ('-/-', ''), 'ru': ('-/-', '')}},")

In [20]:
dd_provinces_replacement = {
    'Canada': {'en': ('Canada', ''), 'ru': ('Канада', '')},
    'Alberta': {'en': ('Alberta', 'Alberta'), 'ru': ('Альберта', 'Альберта')},
    'British Columbia': {'en': ('British Columbia', 'British Columbia'), 'ru': ('Британская Колумбия', 'Британская Колумбия')},
    'Manitoba': {'en': ('Manitoba', 'Manitoba'), 'ru': ('Манитоба', 'Манитоба')},
    'New Brunswick': {'en': ('New Brunswick', 'New Brunswick'), 'ru': ('Нью-Брансуик', 'Нью-Брансуик')},
    'Newfoundland and Labrador': {'en': ('Newfoundland and Labrador', 'Newfoundland and Labrador'), 'ru': ('Ньюфаундленд и Лабрадор', 'Ньюфаундленд и Лабрадор')},
    'Northwest Territories': {'en': ('Northwest Territories', 'Northwest Territories'), 'ru': ('Северо-Западные территории', 'Северо-Западные территории')},
    'Nova Scotia': {'en': ('Nova Scotia', 'Nova Scotia'), 'ru': ('Новая Шотландия', 'Новая Шотландия')},
    'Nunavut': {'en': ('Nunavut', 'Nunavut'), 'ru': ('Нунавут', 'Нунавут')},
    'Ontario': {'en': ('Ontario', 'Ontario'), 'ru': ('Онтарио', 'Онтарио')},
    'Prince Edward Island': {'en': ('Prince Edward Island', 'Prince Edward Island'), 'ru': ('Остров Принца Эдуарда', 'Остров Принца Эдуарда')},
    'Quebec': {'en': ('Quebec', 'Quebec'), 'ru': ('Квебек', 'Квебек')},
    'Saskatchewan': {'en': ('Saskatchewan', 'Saskatchewan'), 'ru': ('Саскачеван', 'Саскачеван')},
    'Yukon': {'en': ('Yukon', 'Yukon'), 'ru': ('Юкон', 'Юкон (территория)')},
}

In [21]:
dd_health_region = {
    'Richmond Health Service Delivery Area': {'en': ('Richmond', 'Richmond, British Columbia'), 'ru': ('Ри́чмонд', 'Ричмонд (Британская Колумбия)')},
    'York Regional Health Unit': {'en': ('York', 'Regional Municipality of York'), 'ru': ('Йорк', 'Йорк (Онтарио)')},
    'Peel Regional Health Unit': {'en': ('Peel', 'Regional Municipality of Peel'), 'ru': ('Пил', 'Пил (район)')},
    'Vancouver Health Authority': {'en': ('Vancouver', 'Vancouver'), 'ru': ('Ванку́вер ', 'Ванкувер')},
    'Halton Regional Health Unit': {'en': ('Halton', 'Regional Municipality of Halton'), 'ru': ('Холтон', '-')},
    'City of Toronto Health Unit': {'en': ('Toronto', 'Toronto'), 'ru': ('Торо́нто', 'Торонто')}
}

In [22]:
# create code for placing info in Wikipedia
def create_table_regions(df, file_header, lang='ru'):

    def if_value(x, prec=1):
        return '—' if math.isnan(x) else \
               f"{x:0.{prec}f}"  if x>=0 else \
               f"−{-x:0.{prec}f}"                #"{x:0.{prec}f}".format(x, prec)
    
    def chval(x, prec=1, *, add_par=''):  # change_value
        return f'style="{add_par}"| —' if math.isnan(x) else \
               f'style="color:darkgreen;{add_par}"| {x:0.{prec}f}' if x>0 else \
               f'style="color:crimson;{add_par}"| −{-x:0.{prec}f}' if x<0 else \
               f'style="color:darkgray;{add_par}"| {x:0.{prec}f}'
    
    def chval_bold(x, prec=1, *, add_par=''):  # change_value
        return f'style="{add_par}"| \'\'\'—\'\'\'' if math.isnan(x) else \
               f'style="color:darkgreen;{add_par}"| \'\'\'{x:0.{prec}f}\'\'\'' if x>0 else \
               f'style="color:crimson;{add_par}"| \'\'\'−{-x:0.{prec}f}\'\'\'' if x<0 else \
               f'style="color:darkgray;{add_par}"| \'\'\'{x:0.{prec}f}\'\'\''
    

    with open('design/' + file_header, mode='r', encoding="utf-8") as fh:
        table_header = fh.read()
        
    ls_health_regions_known = dd_health_region.keys()

    st = ''
    for i in range(len(df)):
        ser = df.iloc[i]
        province_name_visible = dd_provinces_replacement[ser["province"]][lang][1]
        province_link         = dd_provinces_replacement[ser["province"]][lang][0]
        province_inserted = f'[[{province_name_visible}]]' if province_name_visible == province_link else \
                            f'[[{province_name_visible}|{province_link}]]'
        
        # province_inserted = f'[[{province_name_visible}|{province_link}]]' if ser["province"] != "-" else "—"
        
        if ser.name.endswith(' on average'):
             st += '\n' + '|-class=static-row-header\n' + \
                  f'| \'\'\'{dd_provinces_replacement[ser.name[:-11]][lang][0]} в среднем\'\'\' ' + \
                  f'|| {province_inserted} ' + \
                  f'||style="background:#e0ffd8;"| \'\'\'{if_value(ser["2010-12_t"])}\'\'\' ' + \
                  f'||style="background:#eaf3ff;"| \'\'\'{if_value(ser["2010-12_m"])}\'\'\' ' + \
                  f'||style="background:#fee7f6;"| \'\'\'{if_value(ser["2010-12_f"])}\'\'\' ' + \
                  f'||style="background:#fff8dc;"| \'\'\'{if_value(ser["2010-12_fΔm"])}\'\'\' ' + \
                  f'||{chval_bold(ser["Δ_periods"], add_par="padding-right:4.5ex;border-left-width:2px;")} ' + \
                  f'||style="background:#e0ffd8;border-left-width:2px;"| \'\'\'{if_value(ser["2015-17_t"])}\'\'\' ' + \
                  f'||style="background:#eaf3ff;"| \'\'\'{if_value(ser["2015-17_m"])}\'\'\' ' + \
                  f'||style="background:#fee7f6;"| \'\'\'{if_value(ser["2015-17_f"])}\'\'\' ' + \
                  f'||style="background:#fff8dc;"| \'\'\'{if_value(ser["2015-17_fΔm"])}\'\'\''
        else:
            name_inserted = ser.name
            if (name_inserted in ls_health_regions_known) and (dd_health_region[name_inserted][lang][1] != '-'):
                name_inserted = f"[[{dd_health_region[name_inserted][lang][1]}|{name_inserted}]]"
                pass
            
            st += '\n' + '|-\n' + \
                  f'| {name_inserted} ' + \
                  f'|| {province_inserted} ' + \
                  f'||style="background:#e0ffd8;"| \'\'\'{if_value(ser["2010-12_t"])}\'\'\' ' + \
                  f'||style="background:#eaf3ff;"| {if_value(ser["2010-12_m"])} ' + \
                  f'||style="background:#fee7f6;"| {if_value(ser["2010-12_f"])} ' + \
                  f'||style="background:#fff8dc;"| {if_value(ser["2010-12_fΔm"])} ' + \
                  f'||{chval(ser["Δ_periods"], add_par="padding-right:4.5ex;border-left-width:2px;")} ' + \
                  f'||style="background:#e0ffd8;border-left-width:2px;"| \'\'\'{if_value(ser["2015-17_t"])}\'\'\' ' + \
                  f'||style="background:#eaf3ff;"| {if_value(ser["2015-17_m"])} ' + \
                  f'||style="background:#fee7f6;"| {if_value(ser["2015-17_f"])} ' + \
                  f'||style="background:#fff8dc;"| {if_value(ser["2015-17_fΔm"])}'
    if lang == 'ru':
        st = re.sub('(?<=\\d)\\.(?=\\d)', ',', st)  # replace . to comma, if this . is between two digits
        st = st.replace('padding-right:1,5ex;', 'padding-right:1.5ex;') \
               .replace('padding-right:4,5ex;', 'padding-right:4.5ex;')

    st = table_header + st + '\n|}'
    
    # gray color for missing values
    st = st.replace(';"| —', ';color:silver;"| —') \
           .replace(';"| \'\'\'—', ';color:silver;"| \'\'\'—')

    return st


table_code = create_table_regions(df_regions, file_header='Canada_header_health_regions_ru.txt', lang='ru')

# write the code to file
with open('output/Table code for Canadian health regions -ru.txt', 'w', encoding="utf-8") as fh:
    fh.write(table_code)